In [ ]:
%env  WORKDIR=/tmp/vault

In [ ]:
%%bash
cat ${WORKDIR}/cluster-keys.json | jq -r ".root_token"

In [ ]:
%env VAULT_TOKEN=<set-your-token>
%env VAULT_ADDR=https://127.0.0.1:8443
%env VAULT_CACERT=/tmp/vault/vault.ca

In [ ]:
%%bash
vault secrets enable -path=secret kv-v2
vault kv put secret/webapp/config foo=bar username="static-user" password="static-password"


In [ ]:
%%bash

vault policy write superuser -<<EOF
path "*" {
  capabilities = ["create", "read", "update", "delete", "list", "sudo"]
}
EOF


In [ ]:
%%bash
vault auth enable userpass
vault write auth/userpass/users/tester password="changeme" policies="superuser"

In [ ]:
%%bash 
vault write -f sys/replication/primary/enable \
    primary_cluster_addr=https://vault-active.vault.svc.cluster.local:8201

In [ ]:
%%bash 
vault write -format=json sys/replication/performance/primary/secondary-token id=prp12 | jq -r .wrap_info.token > ${WORKDIR}/pr_token.txt
cat ${WORKDIR}/pr_token.txt

## Secondary

In [ ]:
%%bash
cat ${WORKDIR}/clusterpr-keys.json | jq -r ".root_token"

In [ ]:
%env VAULT_TOKEN=<set-your-token>
%env VAULT_ADDR=https://127.0.0.1:8300

In [ ]:
%%bash
vault write sys/replication/performance/secondary/enable  \
    primary_api_addr=https://vault-active.vault.svc.cluster.local:8200 \
    ca_file=/vault/userconfig/vault-ha-tls/vault.ca \
    token=$(cat ${WORKDIR}/pr_token.txt)

In [ ]:
## Unseal other nodes with primary unseal key

In [ ]:
%%bash
kubectl exec -n vaultpr vaultpr-1 -- vault operator unseal $(jq -r ".unseal_keys_b64[]" ${WORKDIR}/cluster-keys.json)

In [ ]:
%%bash
kubectl exec -n vaultpr vaultpr-2 -- vault operator unseal $(jq -r ".unseal_keys_b64[]" ${WORKDIR}/cluster-keys.json)

In [ ]:
%%bash
kubectl exec vaultpr-0 -n vaultpr -- vault read sys/replication/performance/status

In [ ]:
%%bash
kubectl exec vaultpr-1 -n vaultpr -- vault read sys/replication/performance/status

### Checking what happens when you create a token on the primary and want to verify if I can access a PR

In [ ]:
%%bash
kubectl exec vault-0 -n vault -- vault login $(jq -r ".root_token" ${WORKDIR}/cluster-keys.json)
kubectl exec vault-0 -n vault -- vault token create -policy=superuser

In [ ]:
%%bash
echo "Use a short-lived token generated in your environment"
echo "kubectl exec vault-0 -n vault -- vault login <token>"
echo "kubectl exec vaultpr-0 -n vaultpr -- vault login <token>"
